In [1]:
import os
import certifi
import requests
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool

In [2]:
#Agent related functionality
from langchain.agents import AgentExecutor, create_react_agent # reasoning and acting agent

In [3]:
# =============================
# LOAD ENVIRONMENT VARIABLES
# =============================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY= os.getenv("WEATHERSTACK_API_KEY")

os.environ["SSL_CERT_FILE"] = certifi.where()

is basically telling Python:

🔐 "Use this trusted certificate file when making HTTPS/SSL connections."

In [4]:
search_tool = TavilySearchResults(
    tavily_api_key=TAVILY_API_KEY,
    search_engine="google",
    max_results=2,
    return_direct_answer=True,
)

In [5]:
@tool
def get_weather_data(city:str)->str:
    """ 
    Get weather data for a given city.
    """

    url=(
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response=requests.get(url)
    data= response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Description: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%\n"
    )

In [6]:
result=search_tool.invoke("Who is the Current CM of tamilnadu?")
result

[{'url': 'https://en.wikipedia.org/wiki/Chief_Minister_of_Tamil_Nadu',
  'content': '| Tamiḻnāṭ Mutalamaiccar |\n| Emblem of Tamil Nadu |\n| Flag of India |\n| Incumbent C. Joseph Vijay since 10 May 2026 |\n| Style "Style (form of address)") | The Honourable |\n| Type | Head of government |\n| Abbreviation | CM |\n| Member of |  Tamil Nadu Legislative Assembly  Tamil Nadu Council of Ministers |\n| Reports to |  Governor of Tamil Nadu  Tamil Nadu Legislative Assembly |\n| Appointer | Governor of Tamil Nadu |\n| Formation | 10 April 1952; 74 years ago (1952-04-10) |\n| First holder |  A. Subbarayalu Reddiar (as Chief Minister of the Madras Presidency)  P. S. Kumaraswamy Raja (as Chief Minister of Madras State)  C. N. Annadurai (as Chief Minister of Tamil Nadu) |\n| Deputy | Deputy Chief Minister of Tamil Nadu |\n| Website | tn.gov.in/con\\_cmsc.php | [...] ## External links\n\n[edit]\n\n Official Website of the Office of the Chief Minister\n\n|  v  t  e  Chief Minister of Tamil Nadu |\n\

In [7]:
contents = "\n\n".join(
    item["content"] for item in result
)
contents

'| Tamiḻnāṭ Mutalamaiccar |\n| Emblem of Tamil Nadu |\n| Flag of India |\n| Incumbent C. Joseph Vijay since 10 May 2026 |\n| Style "Style (form of address)") | The Honourable |\n| Type | Head of government |\n| Abbreviation | CM |\n| Member of |  Tamil Nadu Legislative Assembly  Tamil Nadu Council of Ministers |\n| Reports to |  Governor of Tamil Nadu  Tamil Nadu Legislative Assembly |\n| Appointer | Governor of Tamil Nadu |\n| Formation | 10 April 1952; 74 years ago (1952-04-10) |\n| First holder |  A. Subbarayalu Reddiar (as Chief Minister of the Madras Presidency)  P. S. Kumaraswamy Raja (as Chief Minister of Madras State)  C. N. Annadurai (as Chief Minister of Tamil Nadu) |\n| Deputy | Deputy Chief Minister of Tamil Nadu |\n| Website | tn.gov.in/con\\_cmsc.php | [...] ## External links\n\n[edit]\n\n Official Website of the Office of the Chief Minister\n\n|  v  t  e  Chief Minister of Tamil Nadu |\n\n| Madras |  P. S. Kumaraswamy Raja  C. Rajagopalachari  K. Kamaraj  M. Bhaktavatsal

### Different temperatures

| Temperature | Behavior | Good for |
|---:|---|---|
| `0.0` | 🎯 Very focused / consistent | Q&A, agents, coding |
| `0.3` | 📌 Slight variation | General conversation |
| `0.7` | 💡 More creative | Writing, brainstorming |
| `1.0` | 🎨 More varied / unpredictable | Creative tasks |

In [8]:
# ===========================
# LLM
# ===========================

llm = ChatOpenAI(
    model_name="gpt-4o",
    openai_api_key=OPENAI_API_KEY,
    temperature=0.0,
    max_tokens=1000,
)

In [9]:
response=llm.invoke("What year is it now?")
response

AIMessage(content="I'm unable to provide real-time information, but my training data goes up to October 2023. You would need to check a current source to find out the present year.", response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 13, 'total_tokens': 48, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_ffd8308b42', 'finish_reason': 'stop', 'logprobs': None}, id='run-900b1bd1-8fe3-4179-933c-14a698156793-0')

In [10]:
# =========================
# PROMPT
# =========================
prompt = hub.pull(
    "hwchase17/react",
)
prompt

c:\Robin\AI course\complete-agentic-ai\.venv\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [11]:
# =========================
# TOOLS
# =========================

tools=[search_tool,get_weather_data]

In [12]:
# =========================
# CREATE AGENT
# =========================

from langchain.agents import AgentExecutor, create_react_agent

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [13]:
# =========================
# Executor
# =========================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

verbose=True tells LangChain to show you what the agent is doing internally while it runs. 🔍

In [14]:
# =========================
# RUN
# =========================

response=agent_executor.invoke({
    "input":(
        "Find the Capital of India"
        "and then find its current weather"
    )
})



> Entering new AgentExecutor chain...
The capital of India is New Delhi. Now, I need to find the current weather in New Delhi.
Action: get_weather_data
Action Input: New DelhiCity: New Delhi
Temperature: 39°C
Description: Sunny
Humidity: 22%
I now know the final answer.

Final Answer: The capital of India is New Delhi. The current weather in New Delhi is 39°C, sunny, with a humidity of 22%.

> Finished chain.


In [15]:
print(response["output"])

The capital of India is New Delhi. The current weather in New Delhi is 39°C, sunny, with a humidity of 22%.
